# Etapa 3: Modelos Preentrenados y Transformers

**Objetivo**: Aplicar transfer learning con ResNet50 y Vision Transformer (ViT) para clasificación de defectos, comparando con modelos anteriores.

**Contenido**:
1. Setup e importaciones
2. Carga de datos (reuso de splits)
3. ResNet50 — Transfer Learning
4. Vision Transformer (ViT) — Transfer Learning
5. Comparación: MLP vs CNN vs ResNet50 vs ViT
6. Grad-CAM comparativo
7. Conclusiones

## 1. Setup e Importaciones

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data_utils import (
    get_dataloaders, get_transforms, CLASS_NAMES,
    get_class_weights, IMAGENET_MEAN, IMAGENET_STD,
)
from src.models import (
    create_resnet50, create_vit, unfreeze_last_n_layers,
    count_parameters,
)
from src.train import train_classifier
from src.evaluate import (
    compute_metrics, get_predictions_with_proba,
    plot_confusion_matrix, plot_training_history,
    plot_roc_curves, plot_model_comparison, print_comparison_table,
)
from src.gradcam import generate_gradcam, plot_gradcam, get_target_layer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
FIGURES_DIR = Path("../figures/etapa3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

## 2. Carga de Datos

In [ ]:
# Transforms para modelos preentrenados (224x224, normalización ImageNet)
train_transform = get_transforms(img_size=224, augment=True)
val_transform = get_transforms(img_size=224, augment=False)

data_dir = Path("../data/splits")
class_weights = get_class_weights(data_dir / "train")

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=data_dir,
    train_transform=train_transform,
    val_transform=val_transform,
    batch_size=16,  # Batch más pequeño para modelos grandes
    num_workers=2,
)

print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

## 3. ResNet50 — Transfer Learning

ResNet50 preentrenado en ImageNet (~25M params). Estrategia:
1. Congelar backbone, entrenar solo cabeza nueva
2. Descongelar últimas 2 capas (fine-tuning gradual)

In [ ]:
# Fase 1: Solo cabeza (backbone congelado)
resnet_model = create_resnet50(num_classes=5, pretrained=True, freeze_backbone=True)
params_resnet = count_parameters(resnet_model)
print(f"ResNet50 (backbone congelado):")
print(f"  Total:      {params_resnet['total']:>12,}")
print(f"  Entrenables: {params_resnet['trainable']:>12,} ({params_resnet['trainable_pct']:.1f}%)")

# Entrenar fase 1
print("\n--- Fase 1: Entrenar cabeza ---")
history_resnet_ph1 = train_classifier(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=15,
    lr=1e-3,
    class_weights=class_weights,
    device=DEVICE,
    save_dir=MODELS_DIR,
    model_name="resnet50_ph1",
    patience=5,
    scheduler_type="plateau",
)

In [ ]:
# Fase 2: Descongelar últimas capas para fine-tuning
print("--- Fase 2: Fine-tuning (últimas 2 capas + cabeza) ---")
unfreeze_last_n_layers(resnet_model, n=2)
params_resnet_ft = count_parameters(resnet_model)
print(f"Parámetros entrenables tras descongelar: {params_resnet_ft['trainable']:,} ({params_resnet_ft['trainable_pct']:.1f}%)")

history_resnet_ph2 = train_classifier(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=15,
    lr=1e-4,  # LR más bajo para fine-tuning
    class_weights=class_weights,
    device=DEVICE,
    save_dir=MODELS_DIR,
    model_name="resnet50_ph2",
    patience=5,
    scheduler_type="cosine",
)

In [ ]:
# Curvas de entrenamiento ResNet50 (ambas fases)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, hist, title in zip(
    axes,
    [history_resnet_ph1, history_resnet_ph2],
    ["ResNet50 Fase 1 (cabeza)", "ResNet50 Fase 2 (fine-tuning)"]
):
    ax.plot(hist["train_acc"], label="Train Acc")
    ax.plot(hist["val_acc"], label="Val Acc")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_resnet50_history.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Vision Transformer (ViT) — Transfer Learning

ViT-Base preentrenado en ImageNet-21k (google/vit-base-patch16-224, ~86M params).
Estrategia similar: congelar → entrenar cabeza → fine-tuning gradual.

In [ ]:
# Fase 1: Solo cabeza del clasificador
vit_model = create_vit(num_classes=5, freeze_backbone=True)
params_vit = count_parameters(vit_model)
print(f"ViT-Base (backbone congelado):")
print(f"  Total:      {params_vit['total']:>12,}")
print(f"  Entrenables: {params_vit['trainable']:>12,} ({params_vit['trainable_pct']:.1f}%)")

print("\n--- Fase 1: Entrenar cabeza ---")
history_vit_ph1 = train_classifier(
    model=vit_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=15,
    lr=1e-3,
    class_weights=class_weights,
    device=DEVICE,
    save_dir=MODELS_DIR,
    model_name="vit_ph1",
    patience=5,
    scheduler_type="plateau",
    is_hf_model=True,
)

In [ ]:
# Fase 2: Fine-tuning (descongelar últimas 2 capas del encoder)
print("--- Fase 2: Fine-tuning ViT ---")
unfreeze_last_n_layers(vit_model, n=2)
params_vit_ft = count_parameters(vit_model)
print(f"Parámetros entrenables tras descongelar: {params_vit_ft['trainable']:,} ({params_vit_ft['trainable_pct']:.1f}%)")

history_vit_ph2 = train_classifier(
    model=vit_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=10,
    lr=2e-5,  # LR muy bajo para ViT fine-tuning
    class_weights=class_weights,
    device=DEVICE,
    save_dir=MODELS_DIR,
    model_name="vit_ph2",
    patience=5,
    scheduler_type="cosine",
    is_hf_model=True,
)

In [ ]:
# Curvas ViT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, hist, title in zip(
    axes,
    [history_vit_ph1, history_vit_ph2],
    ["ViT Fase 1 (cabeza)", "ViT Fase 2 (fine-tuning)"]
):
    ax.plot(hist["train_acc"], label="Train Acc")
    ax.plot(hist["val_acc"], label="Val Acc")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_vit_history.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Evaluación y Comparación Global

### 5.1 Métricas en Test Set

In [ ]:
# Evaluación ResNet50 en test
resnet_model.eval()
y_true_r, y_pred_r, y_proba_r = get_predictions_with_proba(resnet_model, test_loader, DEVICE)
metrics_resnet = compute_metrics(y_true_r, y_pred_r, y_proba_r, CLASS_NAMES)

# Evaluación ViT en test
vit_model.eval()
y_true_v, y_pred_v, y_proba_v = get_predictions_with_proba(vit_model, test_loader, DEVICE, is_hf_model=True)
metrics_vit = compute_metrics(y_true_v, y_pred_v, y_proba_v, CLASS_NAMES)

# Cargar métricas previas
metrics_mlp = json.load(open(MODELS_DIR / "mlp_base_metrics.json")) if (MODELS_DIR / "mlp_base_metrics.json").exists() else {"accuracy": 0, "f1_weighted": 0, "roc_auc_ovr": 0}
metrics_cnn = json.load(open(MODELS_DIR / "cnn_best_metrics.json")) if (MODELS_DIR / "cnn_best_metrics.json").exists() else {"accuracy": 0, "f1_weighted": 0, "roc_auc_ovr": 0}

# Tabla comparativa completa
all_metrics = {
    "MLP Base": metrics_mlp,
    "Mejor CNN": metrics_cnn,
    "ResNet50": metrics_resnet,
    "ViT-Base": metrics_vit,
}
print_comparison_table(all_metrics)

In [ ]:
# Gráfico comparativo
metric_names = ["accuracy", "f1_weighted", "roc_auc_ovr"]
plot_model_comparison(all_metrics, metric_names, save_path=FIGURES_DIR / "03_all_models_comparison.png")

### 5.2 Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ResNet50
plot_confusion_matrix(y_true_r, y_pred_r, CLASS_NAMES, title="ResNet50",
                      save_path=FIGURES_DIR / "03_resnet50_cm.png", ax=axes[0])

# ViT
plot_confusion_matrix(y_true_v, y_pred_v, CLASS_NAMES, title="ViT-Base",
                      save_path=FIGURES_DIR / "03_vit_cm.png", ax=axes[1])

plt.tight_layout()
plt.show()

## 6. Grad-CAM Comparativo

Comparamos la atención de ResNet50 vs ViT para las mismas imágenes.

In [ ]:
# Seleccionar muestras (1 por clase)
test_dataset = test_loader.dataset
samples = {}
for idx in range(len(test_dataset)):
    img, label = test_dataset[idx]
    if label not in samples:
        samples[label] = img
    if len(samples) == len(CLASS_NAMES):
        break

# Grad-CAM para ResNet50
target_layer_resnet = get_target_layer(resnet_model, model_type="resnet50")
# Grad-CAM para ViT
target_layer_vit = get_target_layer(vit_model, model_type="vit")

n_classes = len(samples)
fig, axes = plt.subplots(n_classes, 3, figsize=(12, 4 * n_classes))

for i, (label_id, img_tensor) in enumerate(sorted(samples.items())):
    # Imagen original
    img_show = img_tensor.clone()
    for c in range(3):
        img_show[c] = img_show[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
    img_np = img_show.permute(1, 2, 0).numpy().clip(0, 1)

    # Grad-CAM ResNet50
    cam_resnet = generate_gradcam(resnet_model, img_tensor.unsqueeze(0), target_layer_resnet, device=DEVICE)

    # Grad-CAM ViT
    cam_vit = generate_gradcam(vit_model, img_tensor.unsqueeze(0), target_layer_vit,
                                device=DEVICE, is_vit=True)

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f"{CLASS_NAMES[label_id]}", fontsize=11)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cam_resnet)
    axes[i, 1].set_title("ResNet50 Grad-CAM")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(cam_vit)
    axes[i, 2].set_title("ViT Grad-CAM")
    axes[i, 2].axis("off")

plt.suptitle("Grad-CAM: ResNet50 vs ViT", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_gradcam_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Guardar Resultados

In [ ]:
# Guardar métricas de ResNet50 y ViT
for name, metrics in [("resnet50", metrics_resnet), ("vit", metrics_vit)]:
    path = MODELS_DIR / f"{name}_metrics.json"
    metrics_json = {k: float(v) if hasattr(v, 'item') else v for k, v in metrics.items()}
    with open(path, 'w') as f:
        json.dump(metrics_json, f, indent=2)
    print(f"✓ Métricas {name} guardadas en {path}")

# Guardar comparación global
all_metrics_save = {}
for name, m in all_metrics.items():
    all_metrics_save[name] = {k: float(v) if isinstance(v, (int, float)) else v for k, v in m.items()}
with open(MODELS_DIR / "all_models_comparison.json", 'w') as f:
    json.dump(all_metrics_save, f, indent=2)
print("✓ Comparación global guardada")

## 8. Conclusiones

### Transfer Learning vs. Entrenar desde Cero
- **ResNet50**: Aprovecha features visuales aprendidas en ImageNet (bordes, texturas, formas)
- **ViT**: Captura relaciones globales mediante self-attention, útil para defectos distribuidos
- Ambos superan significativamente a la CNN entrenada desde cero con datos limitados

### Grad-CAM
- ResNet50 tiende a enfocarse en regiones locales específicas del defecto
- ViT muestra patrones de atención más distribuidos, capturando contexto global

### Comparación de Parámetros
- MLP: ~8M params, sin estructura espacial
- CNN custom: ~500K params, captura features locales
- ResNet50: ~25M params (mayoría congelados), transfer learning efectivo
- ViT: ~86M params (mayoría congelados), atención global

### Próximos Pasos
- **Etapa 4**: Modelos generativos (VAE + GAN) para generar datos sintéticos
- **Etapa 5**: LoRA/PEFT para fine-tuning eficiente, cuantización, pruning y despliegue